# MADIS Module Demo

This notebook demonstrates how to use the `madis` module to download and process precipitation gauge data from NOAA's Meteorological Assimilation Data Ingest System (MADIS).

The module exposes three public functions:
- `download_madis()` — download and decompress a single CRN NetCDF file
- `process_madis()` — load a local NetCDF file and return a filtered GeoDataFrame
- `get_data()` — high-level wrapper that does both for a time range

In [1]:
import sys
import os

# if running from the datamods/ directory, this import works directly
# otherwise, adjust the path below to point to the datamods/ folder
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import madis

# directory where downloaded .nc files will be stored
DATA_DIR = '/netfiles/ciroh/qpeData'

---
## 1. Download a single file

`download_madis()` fetches one MADIS CRN file for a given date/time.
If the file is already on disk, it will be skipped automatically.

In [2]:
nc_file = madis.download_madis(
    date='20251130_0000',
    data_dir=DATA_DIR
)
print(f'Downloaded to: {nc_file}')

Skipping download; 20251130_0000.nc found at: /netfiles/ciroh/qpeData/20251130_0000.nc
Downloaded to: /netfiles/ciroh/qpeData/20251130_0000.nc


---
## 2. Process a single file

`process_madis()` loads a local NetCDF file, extracts `precipAccum`, filters to a bounding box, and returns a GeoDataFrame.

By default the bounding box covers the continental United States (CONUS).

In [3]:
gdf = madis.process_madis(nc_file)
gdf.head()

TASK INITIATED: Process MADIS CRN 20251130_0000.nc...
Filtering to bounding box: {'min_lon': -127.699, 'max_lon': -65.137, 'min_lat': 24.275, 'max_lat': 49.84}


/data/condaShared/envs/forecast/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'latitude' has multiple fill values {np.float64(-9999.0), np.float32(3.4028235e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/data/condaShared/envs/forecast/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'longitude' has multiple fill values {np.float64(-9999.0), np.float32(3.4028235e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/data/condaShared/envs/forecast/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'elevation' has multiple fill values {np.float64(-9999.0), np.float32(3.4028235e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/data/condaShared/envs/forecast/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'dataPlatformType' has multiple fill values {

KeyError: "No variable named 'precipAccum'. Did you mean one of ('precipAccum24h', 'precipAccum24hDD', 'precipAccum24hQCR', 'precipAccum24hQCD', 'precipAccum24hQCA', 'precip5min', 'archivePrecipAccum1h', 'rawPrecipAccumTipBuck', 'precip5minDD', 'archivePrecipAccum1hDD')?"

In [ ]:
print(f'CRS: {gdf.crs}')
print(f'Stations: {len(gdf)}')
print(f'Columns: {list(gdf.columns)}')
print(f'\nprecipAccum summary:')
gdf['precipAccum'].describe()

### Custom bounding box

You can pass any bounding box in WGS84 lat/lon degrees.

In [ ]:
# example: Pacific Northwest region
pnw_bbox = {
    'min_lon': -125.0,
    'max_lon': -110.0,
    'min_lat': 42.0,
    'max_lat': 49.0
}

gdf_pnw = madis.process_madis(nc_file, bbox=pnw_bbox)
print(f'Stations in Pacific Northwest: {len(gdf_pnw)}')
gdf_pnw.head()

---
## 3. Quick plot

Visualize gauge locations and precipitation values.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
gdf.plot(
    ax=ax,
    column='precipAccum',
    cmap='Blues',
    legend=True,
    markersize=5,
    legend_kwds={'label': 'Accumulated Precipitation (mm)'}
)
ax.set_title('MADIS CRN Precipitation Gauges — 2025-11-30 00:00 UTC')
ax.set_axis_off()
plt.tight_layout()
plt.show()

---
## 4. Get data for a time range

`get_data()` is the high-level function. It downloads all hourly files between `start_datetime` and `end_datetime` (inclusive), processes each one, and returns a single merged GeoDataFrame with a `datetime` column.

In [ ]:
gdf_range = madis.get_data(
    start_datetime='20251130_0000',
    end_datetime='20251130_0200',
    data_dir=DATA_DIR
)

print(f'Total rows: {len(gdf_range)}')
print(f'Unique timestamps: {sorted(gdf_range["datetime"].unique())}')
gdf_range.head()

### Filter to a specific timestep from the merged result

In [ ]:
import datetime as dt

ts = dt.datetime(2025, 11, 30, 1, 0)  # 01:00 UTC
gdf_1h = gdf_range[gdf_range['datetime'] == ts]
print(f'Stations at {ts}: {len(gdf_1h)}')
gdf_1h.head()

---
## 5. Export to GeoJSON

Pass `export_path` to `get_data()` to save the merged result directly to a GeoJSON file.

In [ ]:
import tempfile, os

export_path = os.path.join(tempfile.gettempdir(), 'madis_precip_gauges.geojson')

gdf_export = madis.get_data(
    start_datetime='20251130_0000',
    end_datetime='20251130_0000',
    data_dir=DATA_DIR,
    export_path=export_path
)

print(f'Exported {len(gdf_export)} rows to: {export_path}')

### Verify the GeoJSON file can be read back

In [ ]:
import geopandas as gpd

gdf_check = gpd.read_file(export_path)
print(f'Rows: {len(gdf_check)}')
print(f'Columns: {list(gdf_check.columns)}')
print(f'CRS: {gdf_check.crs}')
gdf_check.head()

---
## 6. FTP download (alternative)

Pass `data_source='ftp'` to use NOAA's FTP server instead of HTTPS.
This is useful if the HTTPS endpoint is unavailable.

In [ ]:
nc_file_ftp = madis.download_madis(
    date='20251130_0100',
    data_dir=DATA_DIR,
    data_source='ftp'
)
print(f'FTP download complete: {nc_file_ftp}')

---
## Module constants reference

These constants are available at module level if you need to reference them directly.

In [ ]:
print('CONUS bounding box:')
print(madis.CONUS_BBOX)

print('\nHTTPS URL template:')
print(madis.MADIS_BASE_URL)

print('\nFTP host/path:')
print(madis.MADIS_FTP_HOST, madis.MADIS_FTP_PATH)